In [ ]:
from typing import Iterable
from numbers import Number
import joblib
import json
from sklearn.base import BaseEstimator
import pandas as pd
import numpy as np
from sectionproperties.analysis import Section
from sectionproperties.pre.library import rectangular_hollow_section
from sectionproperties.pre import Material
from sigmaepsilon.math.optimize import BinaryGeneticAlgorithm as BGA
from sigmaepsilon.solid.fourier import NavierBeam, LoadGroup, PointLoad, LineLoad
import matplotlib.pyplot as plt

In [2]:
with open("model_regression.pkl", "rb") as f:
    model_regression: BaseEstimator = joblib.load(f)
    
with open("model_logistic_regression.pkl", "rb") as f:
    model_classification: BaseEstimator = joblib.load(f)
    
with open("config.json", "r") as f:
    config: dict = json.load(f)

In [10]:
# Load section data
section_data = config["section"]
material_params = config["material"]

section_variables = []
section_params = []
ranges = []
default_values = []
for p in section_data["params"].keys():
    section_params.append(p)
    if section_data["params"][p]["variable"]:
        section_variables.append(p)
        default_values.append(section_data["params"][p]["default"])
        ranges.append(section_data["params"][p]["range"])

default_section_params = {p:section_data["params"][p]["default"] for p in section_params}
load_components = ["n", "mxx", "myy", "vx", "vy", "mzz"]
beam_length = 10000.0  # mm
number_of_modes = 100
num_evaluation_points = 100

beam_loads = LoadGroup(
    concentrated=LoadGroup(
        LC1=PointLoad(beam_length / 2, [1.0, 0.0]),
        LC2=PointLoad(beam_length / 2, [0.0, 1.0]),
    ),
    distributed=LoadGroup(
        LC3=LineLoad([0, beam_length], [1.0, 0.0]),
        LC4=LineLoad([beam_length / 2, beam_length], [0.0, 1.0]),
    ),
)

In [ ]:
evaluation_points = np.linspace(0, beam_length, num_evaluation_points)
material = Material(**material_params)


def build_section(section_params: dict) -> Section:
    geom_params = {k:v for k,v in section_params.items() if k in section_data["params"]}
    params = {**default_section_params, **geom_params}
    geom = rectangular_hollow_section(**params, material=material)
    geom.create_mesh(mesh_sizes=section_data["mesh_sizes"])
    sec = Section(geometry=geom)
    sec.calculate_frame_properties()
    return sec


def objective(x: Iterable[Number]) -> float:
    section_params = {k: v for k, v in zip(section_variables, x)}
    try:
        sec = build_section(section_params)
    except Exception:
        return np.inf
    eixx, _, _ = sec.get_eic()
    bernoulli_beam = NavierBeam(beam_length, number_of_modes, EI=eixx)
    analysis_results = bernoulli_beam.linear_static_analysis(evaluation_points, beam_loads)
    df = pd.concat([v.to_pandas()[["MZ", "SY"]] for v in analysis_results.values(deep=True)], ignore_index=True)
    df.columns = ["mxx", "vy"]
    for comp in ["n", "myy", "vx", "mzz"]:
        df[comp] = 0.0
    for k, v in default_section_params.items():
        df[k] = v
    df = df[section_variables + load_components]
    utilization = model_regression.predict(df)
    return utilization.max()


bga = BGA(objective, ranges, length=12, nPop=100, minimize=True)

In [20]:
objective(default_values)

3.207624608598008

In [ ]:
history = [objective(bga.best_phenotype())]

for _ in range(100):
    bga.evolve(1)
    history.append(bga.champion.fittness)

plt.plot(history)
plt.title('History of the best solution')
plt.show()

x = bga.champion.phenotype
fx = bga.champion.fittness

print(f"The minimum value is f(x) = {fx} at  x = {x}.")

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x1041251c0>>
Traceback (most recent call last):
  File "/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 
